# Web Scraping and Preprocessing for Bank of Abyssinia (BOA)

The objective of this notebook is to scrape user reviews from the Google Play Store for the Bank of Abyssinia mobile banking application using the `google-play-scraper` library.


In [1]:
# core libraries
import pandas as pd
import numpy as np
import re
from datetime import datetime

# Google Play scrapping library
from google_play_scraper import app, reviews, Sort

print("All libraries are loaded successfully!")


All libraries are loaded successfully!


# 1: Scrape Google Play Store Reviews

In this section, we collect user reviews for the Commercial Bank of Ethiopia mobile banking application from the Google Play Store.


In [2]:
# The unique identifier for BOA's app on the Google Play Store
BOA_APP_ID = "com.boa.boaMobileBanking"

# Get app metadata (rating, installs, description...)
app_info = app(
    BOA_APP_ID,
    lang='en',    # Language: English
    country='et'  # Country: Ethiopia
)

print("=" * 50)
print("BOA App Info")
print("=" * 50)
print(f"App Title   : {app_info['title']}")
print(f"Current Score: {app_info['score']}")
print(f"Total Ratings: {app_info['ratings']:,}")
print(f"Total Reviews: {app_info['reviews']:,}")
print(f"Installs     : {app_info['installs']}")

BOA App Info
App Title   : BoA Mobile
Current Score: 4.3889456
Total Ratings: 9,235
Total Reviews: 1,461
Installs     : 1,000,000+


In [3]:
# Scrape reviews
print(f"Scraping reviews for BOA...")

result, continuation_token = reviews(
    BOA_APP_ID,
    lang='en',
    country='et',
    sort=Sort.NEWEST,       # Fetch the most recent reviews
    count=400,              # Number of reviews to fetch
    filter_score_with=None  # Fetch all reviews regardless of score
)

print(f"Collected {len(result)} raw reviews")

Scraping reviews for BOA...
Collected 400 raw reviews


In [4]:
# Single raw review sample
print("Keys in a single review:") 
print(list(result[0].keys()))

print("\nFirst raw review (sample):")
for key, value in result[0].items():
    print(f"  {key}: {value}")

Keys in a single review:
['reviewId', 'userName', 'userImage', 'content', 'score', 'thumbsUpCount', 'reviewCreatedVersion', 'at', 'replyContent', 'repliedAt', 'appVersion']

First raw review (sample):
  reviewId: 21bcff26-5b05-485d-958f-4832ec1fac01
  userName: Tsegay Teklay
  userImage: https://play-lh.googleusercontent.com/a/ACg8ocIM0xBSUOBR56D_ua5lA4cc88oCASND4P-Xl64vQENvgmaaYQ=mo
  content: Its Good
  score: 5
  thumbsUpCount: 0
  reviewCreatedVersion: 25.09.03
  at: 2026-05-15 15:01:09
  replyContent: None
  repliedAt: None
  appVersion: 25.09.03


In [6]:
# Extract only relevant fields 
raw_data = []

for r in result:
    raw_data.append({
        'review_id': r.get('reviewId', ''),
        'review'   : r.get('content', ''),
        'rating'   : r.get('score', None),
        'date'     : r.get('at', None),
        'bank'     : 'BOA',
        'source'   : 'Google Play'
    })

# Build a DataFrame
df_raw = pd.DataFrame(raw_data)

print(f"Shape: {df_raw.shape}")
df_raw.head()

Shape: (400, 6)


,review_id,review,rating,date,bank,source
0,21bcff26-5b05-485d-958f-4832ec1fac01,Its Good,5,2026-05-15 15:01:09,BOA,Google Play
1,c5eb7589-59b7-4d72-8aa9-100a703ecaa3,good,5,2026-05-14 21:18:44,BOA,Google Play
2,c3bb042c-844b-4580-98b9-df418622b2fb,it's very good app,5,2026-05-12 11:50:32,BOA,Google Play
3,400ce769-3726-43b2-ac4d-755b3a15f026,this app is good but the speed of app is very ...,2,2026-05-11 18:18:54,BOA,Google Play
4,4d6d2f22-5e71-47be-9cde-a1cf6c9fff93,good,5,2026-05-09 14:41:50,BOA,Google Play


In [7]:
# Basic shape and types
print(f"Total reviews collected: {len(df_raw)}")
print(f"\nColumn dtypes:")
print(df_raw.dtypes)

Total reviews collected: 400

Column dtypes:
review_id               str
review                  str
rating                int64
date         datetime64[us]
bank                    str
source                  str
dtype: object


In [8]:
# Rating distribution
print("Rating distribution:")
rating_counts = df_raw['rating'].value_counts().sort_index(ascending=False)
for rating, count in rating_counts.items():
    bar = '█' * (count // 5)
    print(f"  {int(rating)} stars: {count:>4}  {bar}")

Rating distribution:
  5 stars:  231  ██████████████████████████████████████████████
  4 stars:   32  ██████
  3 stars:   13  ██
  2 stars:   13  ██
  1 stars:  111  ██████████████████████


In [9]:
# Date values and dtype check
print("Sample date values (raw):")
print(df_raw['date'].head(10).to_string())

print(f"\nDate dtype: {df_raw['date'].dtype}")

Sample date values (raw):
0   2026-05-15 15:01:09
1   2026-05-14 21:18:44
2   2026-05-12 11:50:32
3   2026-05-11 18:18:54
4   2026-05-09 14:41:50
5   2026-05-08 13:47:07
6   2026-05-07 10:33:06
7   2026-05-05 11:03:08
8   2026-05-04 14:01:17
9   2026-05-03 13:40:13

Date dtype: datetime64[us]


In [10]:
# Data Quality Audit
print("=" * 50)
print("DATA QUALITY AUDIT")
print("=" * 50)

# Missing Values check
print("\nMissing Values")
print("-" * 30)
missing = df_raw.isnull().sum()
missing_pct = (missing / len(df_raw) * 100).round(2)

for col in df_raw.columns:
    status = f"{missing[col]} missing ({missing_pct[col]}%)" if missing[col] > 0 else "OK"
    print(f"  {col:<15}: {status}")

DATA QUALITY AUDIT

Missing Values
------------------------------
  review_id      : OK
  review         : OK
  rating         : OK
  date           : OK
  bank           : OK
  source         : OK


In [11]:
# Duplicate Reviews check
print("Duplicates")
print("-" * 30)

# Exact duplicates on review text
exact_dupes = df_raw.duplicated(subset=['review']).sum()
print(f"  Exact duplicate reviews : {exact_dupes}")

# Duplicate review IDs
id_dupes = df_raw.duplicated(subset=['review_id']).sum()
print(f"  Duplicate review IDs    : {id_dupes}")

# Empty reviews (also a form of bad data)
empty_reviews = (df_raw['review'].str.strip() == '').sum()
print(f"  Empty review texts      : {empty_reviews}")

Duplicates
------------------------------
  Exact duplicate reviews : 66
  Duplicate review IDs    : 0
  Empty review texts      : 0


In [12]:
# Date Format check
print("Date Format")
print("-" * 30)
print(f"  Current dtype: {df_raw['date'].dtype}")
print(f"  Sample values: {df_raw['date'].iloc[0]}")
print(f"  Target format: YYYY-MM-DD (string or date object)")

Date Format
------------------------------
  Current dtype: datetime64[us]
  Sample values: 2026-05-15 15:01:09
  Target format: YYYY-MM-DD (string or date object)


In [13]:
#copying row data to a new dataframe for cleaning and analysis
df = df_raw.copy()

print(f"Starting with: {len(df)} reviews")

Starting with: 400 reviews


In [14]:
#Handle Missing Values
before = len(df)

# Drop rows missing the critical columns
critical_cols = ['review', 'rating']
df = df.dropna(subset=critical_cols)

removed = before - len(df)
print(f"Removed {removed} rows with missing critical data")
print(f"Remaining: {len(df)} reviews")

Removed 0 rows with missing critical data
Remaining: 400 reviews


In [15]:
#Handle Duplicates
before = len(df)

df = df.drop_duplicates(subset=['review_id'], keep='first')

removed = before - len(df)
print(f"Removed {removed} duplicate reviews")
print(f"Remaining: {len(df)} reviews")

Removed 0 duplicate reviews
Remaining: 400 reviews


In [16]:
# Date normalization
print("Before normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

# Convert to pandas datetime, then format as YYYY-MM-DD string
df['date'] = pd.to_datetime(df['date']).dt.strftime('%Y-%m-%d')

print("\nAfter normalization:")
print(df['date'].head(3).to_string())
print(f"dtype: {df['date'].dtype}")

print(f"\nDate range: {df['date'].min()} to {df['date'].max()}")

Before normalization:
0   2026-05-15 15:01:09
1   2026-05-14 21:18:44
2   2026-05-12 11:50:32
dtype: datetime64[us]

After normalization:
0    2026-05-15
1    2026-05-14
2    2026-05-12
dtype: str

Date range: 2025-06-15 to 2026-05-15


In [17]:
# Text Cleaning
def clean_text(text):           # Standardize review text: collapse whitespace, strip edges.
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\s+', ' ', text)  # collapse multiple spaces/newlines
    text = text.strip()               # remove leading/trailing whitespace
    return text


# Apply to the full column
df['review'] = df['review'].apply(clean_text)

# Remove any reviews that became empty after cleaning
before = len(df)
df = df[df['review'].str.len() > 0]
removed = before - len(df)
print(f"\nRemoved {removed} reviews that were empty after cleaning")


Removed 0 reviews that were empty after cleaning


In [18]:
# Check for out-of-range ratings
invalid_ratings = df[(df['rating'] < 1) | (df['rating'] > 5)]
print(f"Invalid ratings (outside 1–5): {len(invalid_ratings)}")

# Remove them
df = df[(df['rating'] >= 1) & (df['rating'] <= 5)]

# Ensure rating is stored as integer
df['rating'] = df['rating'].astype(int)

print(f"Remaining: {len(df)} reviews")
print(f"Rating dtype: {df['rating'].dtype}")

Invalid ratings (outside 1–5): 0
Remaining: 400 reviews
Rating dtype: int64


In [19]:
# Select only the 5 required columns in the right order
df_clean = df[['review', 'rating', 'date', 'bank', 'source']].copy()

# Sort by date (newest first) for clean presentation
df_clean = df_clean.sort_values('date', ascending=False).reset_index(drop=True)

print("=" * 50)
print("FINAL BOA DATASET")
print("=" * 50)

print(f"Shape: {df_clean.shape}")
df_clean.head(10)

FINAL BOA DATASET
Shape: (400, 5)


,review,rating,date,bank,source
0,Its Good,5,2026-05-15,BOA,Google Play
1,good,5,2026-05-14,BOA,Google Play
2,it's very good app,5,2026-05-12,BOA,Google Play
3,this app is good but the speed of app is very ...,2,2026-05-11,BOA,Google Play
4,good,5,2026-05-09,BOA,Google Play
5,boa the best,5,2026-05-08,BOA,Google Play
6,bank of absiniya is best bank in ethiopian,5,2026-05-07,BOA,Google Play
7,አስተማማኝና ዘመኑን የዋጀ,4,2026-05-05,BOA,Google Play
8,good,5,2026-05-04,BOA,Google Play
9,extremely slow app and unreliable for most pay...,2,2026-05-03,BOA,Google Play


In [20]:
# Save to CSV in the raw data folder 
output_path = '../data/raw/boa_reviews_clean.csv'
df_clean.to_csv(output_path, index=False)

print(f"Saved to: {output_path}")

Saved to: ../data/raw/boa_reviews_clean.csv


In [21]:
print("=" * 55)
print("  PREPROCESSING REPORT — BOA Reviews")
print("=" * 55)

original_count = len(df_raw)
final_count    = len(df_clean)
removed_total  = original_count - final_count
retention_rate = (final_count / original_count * 100)

print(f"\n  Raw reviews collected  : {original_count:>6}")
print(f"  Reviews after cleaning : {final_count:>6}")
print(f"  Reviews removed        : {removed_total:>6}")
print(f"  Data retention rate    : {retention_rate:>5.1f}%")

quality = "EXCELLENT" if retention_rate >= 95 else ("GOOD" if retention_rate >= 90 else "NEEDS ATTENTION")
print(f"  Data quality           : {quality}")

print(f"\n  Date range : {df_clean['date'].min()}  to  {df_clean['date'].max()}")

print("\n  Rating distribution:")
for rating in sorted(df_clean['rating'].unique(), reverse=True):
    count = (df_clean['rating'] == rating).sum()
    pct   = count / final_count * 100
    bar   = '█' * (count // 5)
    print(f"    {rating} stars : {count:>4} ({pct:4.1f}%)  {bar}")

print("\n  Text length stats:")
lengths = df_clean['review'].str.len()
print(f"    Min    : {lengths.min()} characters")
print(f"    Median : {lengths.median():.0f} characters")
print(f"    Max    : {lengths.max()} characters")

print("\n  Columns in final CSV:")
for col in df_clean.columns:
    print(f"    - {col}")

print("\n" + "=" * 55)

  PREPROCESSING REPORT — BOA Reviews

  Raw reviews collected  :    400
  Reviews after cleaning :    400
  Reviews removed        :      0
  Data retention rate    : 100.0%
  Data quality           : EXCELLENT

  Date range : 2025-06-15  to  2026-05-15

  Rating distribution:
    5 stars :  231 (57.8%)  ██████████████████████████████████████████████
    4 stars :   32 ( 8.0%)  ██████
    3 stars :   13 ( 3.2%)  ██
    2 stars :   13 ( 3.2%)  ██
    1 stars :  111 (27.8%)  ██████████████████████

  Text length stats:
    Min    : 1 characters
    Median : 14 characters
    Max    : 492 characters

  Columns in final CSV:
    - review
    - rating
    - date
    - bank
    - source

